# Step 1. Load the three datasets

In [5]:
import pandas as pd

# Cleaned job postings
postings = pd.read_csv("processed_data/IE7374 clean_postings.csv")

# Job ID -> Skill abbreviation
job_skills = pd.read_csv("job_skills.csv")

# Skill abbreviation -> Skill name
skill_mapping = pd.read_csv("skills.csv")

# Step 2. Inspect the datasets

In [7]:
print(postings.shape)
print(job_skills.shape)
print(skill_mapping.shape)

(109431, 9)
(213768, 2)
(35, 2)


In [9]:
postings.head()
job_skills.head()
skill_mapping.head()

,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design
2,ADVR,Advertising
3,PRDM,Product Management
4,DIST,Distribution


# Step 3. Merge job_skills with skill_mapping

In [49]:
job_skills = job_skills.merge(
    skill_mapping,
    on="skill_abr",
    how="left"
)

In [51]:
job_skills.head()

,job_id,skill_abr,skill_name_x,skill_name_y,skill_name
0,3884428798,MRKT,Marketing,Marketing,Marketing
1,3884428798,PR,Public Relations,Public Relations,Public Relations
2,3884428798,WRT,Writing/Editing,Writing/Editing,Writing/Editing
3,3887473071,SALE,Sales,Sales,Sales
4,3887465684,FIN,Finance,Finance,Finance


# Step 4. Check for missing mappings

In [53]:
print(job_skills["skill_name"].isna().sum())

0


# Step 5. Aggregate skills for each job posting

In [55]:
grouped_skills = (
    job_skills
    .groupby("job_id")["skill_name"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
)

# Step 6. Merge the skills into the cleaned postings dataset

In [56]:
training_df = postings.merge(
    grouped_skills,
    on="job_id",
    how="left"
)

In [59]:
training_df.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,description_character_count,description_word_count,skill_name
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,Not specified,2525,358,"Marketing, Sales"
1,1829192,Not specified,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,Not specified,3560,492,Health Care Provider
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,Not specified,460,66,"Management, Manufacturing"
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,Not specified,1594,209,Other
4,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,"Raleigh, NC",Internship,Not specified,4188,578,Project Management


# Step 7. Evaluate skill coverage

In [61]:
print(f"Total postings: {len(training_df):,}")

print(
    f"Postings with skill labels: "
    f"{training_df['skill_name'].notna().sum():,}"
)

print(
    f"Coverage: "
    f"{training_df['skill_name'].notna().mean():.2%}"
)

Total postings: 109,431
Postings with skill labels: 107,781
Coverage: 98.49%


# Step 8. Inspect sample records

In [63]:
training_df[
    [
        "title",
        "description",
        "skill_name"
    ]
].sample(
    5,
    random_state=42
)

,title,description,skill_name
59820,Systems Engineer,Job Description: Unified Communications Engine...,Information Technology
34391,North America Cable Design Director - Energy,Position: NA Cable Design Director Energy Comp...,Engineering
4195,Medical Assistant - Medical Associates of Nort...,2024-62214 About This Practice At Medical Asso...,Other
101924,Mammography Technologist,"At Solis Mammography, our patient-focused cult...",Health Care Provider
104273,Inside Sales Applications Engineer,Inside Sales Applications Engineer Job Miller ...,"Customer Service, Engineering, Sales"


# Step 9. Save the final training dataset

In [65]:
training_df.to_csv(
    "processed_data/training_dataset.csv",
    index=False,
    encoding="utf-8"
)

print("Training dataset saved successfully!")

Training dataset saved successfully!


# Step 10. Split the dataset into training, validation, and test sets

Before fine-tuning the model, split the labeled dataset into training, validation, and test sets. This ensures a consistent evaluation protocol and allows fair comparison between the base model and the fine-tuned model.

Since the project proposal specifies an 80% / 10% / 10% split, we follow the same strategy.

In [80]:
# Keep only records with skill labels
labeled_df = training_df.dropna(subset=["skill_name"]).copy()

# Remove empty labels, if any
labeled_df = labeled_df[
    labeled_df["skill_name"].str.strip() != ""
].copy()

print(f"Labeled records available: {len(labeled_df):,}")

Labeled records available: 107,781


In [82]:
# Shuffle the dataset reproducibly
labeled_df = labeled_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [84]:
# Calculate split boundaries
total_rows = len(labeled_df)

train_end = int(total_rows * 0.80)
validation_end = int(total_rows * 0.90)

# Create train, validation, and test sets
train_df = labeled_df.iloc[:train_end].copy()
val_df = labeled_df.iloc[train_end:validation_end].copy()
test_df = labeled_df.iloc[validation_end:].copy()

In [86]:
print(f"Train:      {len(train_df):,} ({len(train_df) / total_rows:.2%})")
print(f"Validation: {len(val_df):,} ({len(val_df) / total_rows:.2%})")
print(f"Test:       {len(test_df):,} ({len(test_df) / total_rows:.2%})")

Train:      86,224 (80.00%)
Validation: 10,778 (10.00%)
Test:       10,779 (10.00%)


In [88]:
from pathlib import Path

OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    OUTPUT_DIR / "train.csv",
    index=False,
    encoding="utf-8"
)

val_df.to_csv(
    OUTPUT_DIR / "validation.csv",
    index=False,
    encoding="utf-8"
)

test_df.to_csv(
    OUTPUT_DIR / "test.csv",
    index=False,
    encoding="utf-8"
)

print("Train, validation, and test datasets saved successfully.")

Train, validation, and test datasets saved successfully.
